# Driving a lead through a funnel, studied

`dev.ipynb` built the core mechanism: discover a skill, load it, run a
plan/act/respond tool loop for one request. This notebook is a different
shape of problem on top of that same mechanism — a marketing/sales rep
agent doesn't answer one request, it **drives a lead through a multi-stage
funnel across many conversation turns**, remembering where each lead is
between messages, and deciding when (and whether) to advance them.

Three new pieces, all in `src/bro_agent/`:

- **`leads.py`** — a SQLite-backed lead record (stage, extracted fields,
  full turn-by-turn history) that persists *between* separate calls, unlike
  the `dev.ipynb` `state` dict which died when `execute_skill` returned.
- **Five funnel skills** in `bro_skills/` (`qualify-lead`, `nurture-lead`,
  `present-offer`, `handle-objection`, `close-handoff`), each declaring a
  new frontmatter field, `stage`, naming which funnel stage it handles.
- **`funnel.py`** — `handle_turn(lead_id, message)`: processes one inbound
  customer message, scoped to whatever skill owns the lead's *current*
  stage, then asks the model to propose a next stage — which gets
  **clamped against an explicit allowed-transitions map** before anything
  is persisted. That clamp is the guardrail: no lead can skip straight to
  `close_handoff` no matter what the model or the customer suggests, and
  `close_handoff` never does anything but create a handoff for a human —
  no autonomous payment or contract action.

The rest of this notebook is mock data (a fake product, fake pricing) and
five scripted conversations, one per scenario, so the guardrails and stage
logic are actually visible doing their job rather than just described.

In [1]:
from bro_agent.skills import discover_skills
from bro_agent.leads import init_db, create_lead, get_lead, DEFAULT_DB_PATH
from bro_agent.funnel import handle_turn, TRANSITIONS

DB_PATH = DEFAULT_DB_PATH
init_db(DB_PATH)

registry = discover_skills()
{name: info["stage"] for name, info in registry.items() if info["stage"]}

{'close-handoff': 'close_handoff',
 'handle-objection': 'handle_objection',
 'nurture-lead': 'nurture',
 'present-offer': 'present_offer',
 'qualify-lead': 'qualify'}

## Mock data

The product/pricing facts the agent grounds itself in already live inside
the skills themselves: `bro_skills/nurture-lead/product_info.md` (a fake
workflow-automation product) and `bro_skills/present-offer/pricing.json`
(`starter`/`pro`/`enterprise`). Nothing else to seed there.

A small helper below creates a fresh mock lead and runs a scripted list of
customer messages through `handle_turn`, one per turn, printing the
stage-before → stage-after and the reply for each — this is what every
scenario cell below calls.

In [2]:
def run_scenario(lead_id: str, contact: str, messages: list[str]) -> dict:
    create_lead(lead_id, contact, path=DB_PATH)
    print(f"=== {lead_id} ({contact}) ===")
    for message in messages:
        before = get_lead(lead_id, path=DB_PATH)["stage"]
        reply = handle_turn(lead_id, message, registry=registry, db_path=DB_PATH)
        after = get_lead(lead_id, path=DB_PATH)["stage"]
        print(f"[{before} -> {after}] customer: {message}")
        print(f"           agent: {reply}\n")
    return get_lead(lead_id, path=DB_PATH)

## Scenario 1 — happy path

A well-qualified lead walks straight through the whole funnel:
`qualify → nurture → present_offer → close_handoff`, and the price quoted
should match `pricing.json`.

In [3]:
happy_lead = run_scenario(
    "demo-happy",
    "Alex <alex@example.com>",
    [
        "hi, looking for a workflow automation tool for my 8-person team",
        "budget is around $150/month, and we'd want to get started within the month",
        "what's the pricing on the pro plan?",
        "sounds good, let's move forward",
    ],
)
print("final stage:", happy_lead["stage"])
print("fields:", happy_lead["fields"])
assert happy_lead["stage"] == "close_handoff", happy_lead["stage"]

=== demo-happy (Alex <alex@example.com>) ===
[qualify -> qualify] customer: hi, looking for a workflow automation tool for my 8-person team
           agent: That’s great! Can you tell me a bit more about the specific workflows you’re hoping to automate, and roughly what your budget is for something like this?

[qualify -> qualify] customer: budget is around $150/month, and we'd want to get started within the month
           agent: Okay, that’s helpful to know! And when you say ‘within the month,’ are you thinking ideally next week, or is that a flexible timeframe?

[qualify -> qualify] customer: what's the pricing on the pro plan?
           agent: The Pro plan is $250/month. To help me understand your needs better, could you tell me what kind of workflows you’re primarily looking to automate?

[qualify -> nurture] customer: sounds good, let's move forward
           agent: Fantastic! To help me narrow down the best options for your team, could you describe a couple of the workflows 

AssertionError: nurture

## Scenario 2 — objection loop

A price objection should send the lead to `handle_objection` and then
*back* to `present_offer` — never forward to `close_handoff` until the
objection is actually resolved.

In [4]:
objection_lead = run_scenario(
    "demo-objection",
    "Sam <sam@example.com>",
    [
        "hi, we need something for automating reports, team of 5",
        "budget's tight, maybe $50/month, need it soon",
        "what would you recommend, pricing-wise?",
        "that's too expensive for us",
        "what if I only need the basics, is there a cheaper option?",
    ],
)
print("final stage:", objection_lead["stage"])
assert objection_lead["stage"] != "close_handoff", "should not have reached close yet"

=== demo-objection (Sam <sam@example.com>) ===
[qualify -> qualify] customer: hi, we need something for automating reports, team of 5
           agent: Great! Could you tell me roughly what kind of reports you're automating and what your budget looks like for this?

[qualify -> qualify] customer: budget's tight, maybe $50/month, need it soon
           agent: Okay, I understand. And roughly, how quickly are you hoping to get this up and running – is that within the next few weeks, or further out?

[qualify -> nurture] customer: what would you recommend, pricing-wise?
           agent: Got it. Considering a team of five and a budget around $50 a month, I can definitely explore some options for you.



ValueError: unknown tool for this stage: handle_objection

## Scenario 3 — premature-close guardrail

This is the scenario that specifically proves the `TRANSITIONS` clamp
works, not just the happy path: a brand-new lead demands to close
immediately. `qualify`'s allowed transitions are only `{qualify, nurture}`
— `close_handoff` isn't reachable from here no matter what either side
says.

In [5]:
premature_lead = run_scenario(
    "demo-premature",
    "Jordan <jordan@example.com>",
    ["just sign me up right now, I want to buy"],
)
print("final stage:", premature_lead["stage"])
assert premature_lead["stage"] == "qualify", premature_lead["stage"]

=== demo-premature (Jordan <jordan@example.com>) ===
[qualify -> qualify] customer: just sign me up right now, I want to buy
           agent: Okay, that's great enthusiasm! Before we talk about getting you set up, could you tell me a little bit more about what you're hoping to achieve and roughly how many people would be involved with this?

final stage: qualify


## Scenario 4 — pricing-grounding check

Once at `present_offer`, the customer asks for a number that doesn't exist
in `pricing.json`. Read the printed reply below: it should not invent a
discount — `present-offer/SKILL.md` explicitly instructs the model to
defer to a human rather than make up a number.

In [6]:
pricing_lead = run_scenario(
    "demo-pricing",
    "Riley <riley@example.com>",
    [
        "hi, team of 3, evaluating a couple of tools",
        "budget roughly $40/month, timeline is this quarter",
        "what plans do you have?",
        "can you do it for $10/month instead?",
    ],
)
print("final stage:", pricing_lead["stage"])
# Eyeball the printed replies above: no $10/month (or any invented number)
# should appear as an agreed price -- only real pricing.json values or a
# deferral to a human.

=== demo-pricing (Riley <riley@example.com>) ===
[qualify -> qualify] customer: hi, team of 3, evaluating a couple of tools
           agent: Okay, great! Could you tell me a little more about the specific problem you're hoping to solve with a new tool?

[qualify -> qualify] customer: budget roughly $40/month, timeline is this quarter
           agent: Got it. And roughly how many people would be using the tool on a regular basis?

[qualify -> qualify] customer: what plans do you have?
           agent: Okay, and what kind of plans are you looking for in a tool – are you thinking about different user roles, features, or something else?

[qualify -> nurture] customer: can you do it for $10/month instead?
           agent: That’s definitely doable! Just to confirm, you’re looking at approximately $10 per month, correct?

final stage: nurture


## Scenario 5 — unqualified / no fit

A lead with no real budget or need should stay in `qualify`/`nurture` —
never reach `present_offer`, since there's nothing to present yet.

In [7]:
unqualified_lead = run_scenario(
    "demo-unqualified",
    "Casey <casey@example.com>",
    ["I just have one user and basically no budget, just curious what this is"],
)
print("final stage:", unqualified_lead["stage"])
assert unqualified_lead["stage"] in {"qualify", "nurture"}, unqualified_lead["stage"]

=== demo-unqualified (Casey <casey@example.com>) ===
[qualify -> qualify] customer: I just have one user and basically no budget, just curious what this is
           agent: Okay, I understand. So you're exploring this for yourself right now. Could you tell me a little bit about the specific challenge you're hoping to address?

final stage: qualify


## Audit trail

Every turn (customer message, agent reply, stage at that point) was
appended to the lead's `history` in SQLite as it happened — the same data
is inspectable directly, e.g. from a terminal:
`sqlite3 data/leads.db "select id, stage from leads"`. Here it is for the
happy-path lead, confirming the printed trace above matches what's
actually persisted.

In [8]:
import json

for turn in get_lead("demo-happy", path=DB_PATH)["history"]:
    print(f"[{turn['stage']:>16}] {turn['role']:>8}: {turn['text']}")

[         qualify] customer: hi, looking for a workflow automation tool for my 8-person team
[         qualify]    agent: That’s great! Can you tell me a bit more about the specific workflows you’re hoping to automate, and roughly what your budget is for something like this?
[         qualify] customer: budget is around $150/month, and we'd want to get started within the month
[         qualify]    agent: Okay, that’s helpful to know! And when you say ‘within the month,’ are you thinking ideally next week, or is that a flexible timeframe?
[         qualify] customer: what's the pricing on the pro plan?
[         qualify]    agent: The Pro plan is $250/month. To help me understand your needs better, could you tell me what kind of workflows you’re primarily looking to automate?
[         qualify] customer: sounds good, let's move forward
[         nurture]    agent: Fantastic! To help me narrow down the best options for your team, could you describe a couple of the workflows you're most 